In [1]:
pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# 1. Simulate Locations (Latitude/Longitude)
# Let's pretend we have 20 delivery spots in Tokyo
def create_data_model():
    """Stores the data for the problem."""
    data = {}
    np.random.seed(42)
    data['locations'] = np.random.randint(0, 100, size=(20, 2)).tolist()
    data['depot'] = 0
    data['num_vehicles'] = 4
    return data

def compute_distance_matrix(locations):
    size = len(locations)
    matrix = {}
    for from_node in range(size):
        matrix[from_node] = {}
        for to_node in range(size):
            x1, y1 = locations[from_node]
            x2, y2 = locations[to_node]
            matrix[from_node][to_node] = abs(x1 - x2) + abs(y1 - y2)
    return matrix

data = create_data_model()
distance_matrix = compute_distance_matrix(data['locations'])

print(f"✅ Environment Ready.")
print(f"We have {len(data['locations'])} locations and {data['num_vehicles']} trucks.")
print(f"Distance from Warehouse (0) to Point 1: {distance_matrix[0][1]} km (approx)")

✅ Environment Ready.
We have 20 locations and 4 trucks.
Distance from Warehouse (0) to Point 1: 58 km (approx)


In [ ]:
# args: (number of locations, number of vehicles, starting node)
manager = pywrapcp.RoutingIndexManager(
    len(data['locations']), 
    data['num_vehicles'], 
    data['depot']
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# --- THE 2024 PROBLEM LOGIC ---
# We add a "Dimension" to track distance.
# We set a HARD CAP (capacity) of 500 units per vehicle.
# If a route requires 501 units, the solver will NOT allow it.
dimension_name = 'Distance'
routing.AddDimension(
    transit_callback_index,
    0,      # no slack (waiting time) allowed at start
    500,    # <--- THIS IS THE LEGAL CAP (e.g., Max Work Hours converted to Distance)
    True,   # start cumul to zero
    dimension_name
)
distance_dimension = routing.GetDimensionOrDie(dimension_name)

distance_dimension.SetGlobalSpanCostCoefficient(100)
print("heijunka (leveling) logic applied")

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

print("🧩 Solving for optimized routes...")
solution = routing.SolveWithParameters(search_parameters)

def print_solution(data, manager, routing, solution):
    print(f'Objective: {solution.ObjectiveValue()} total distance')
    max_route_distance = 0
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Truck {vehicle_id} Route:\n'
        route_distance = 0
        
        while not routing.IsEnd(index):
            plan_output += f' {manager.IndexToNode(index)} ->'
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
            
        plan_output += f' {manager.IndexToNode(index)}\n'
        plan_output += f'Distance: {route_distance}km\n'
        print(plan_output)
        max_route_distance = max(route_distance, max_route_distance)
        
    print(f'Longest Route: {max_route_distance}km')

if solution:
    print_solution(data, manager, routing, solution)
else:
    print("❌ No solution found! The constraints are too strict (The 2024 Crisis).")

heijunka (leveling) logic applied
🧩 Solving for optimized routes...
Objective: 24350 total distance
Truck 0 Route:
 0 -> 1 -> 7 -> 18 -> 10 -> 8 -> 16 -> 0
Distance: 194km

Truck 1 Route:
 0 -> 6 -> 9 -> 12 -> 0
Distance: 236km

Truck 2 Route:
 0 -> 5 -> 3 -> 0
Distance: 98km

Truck 3 Route:
 0 -> 17 -> 4 -> 15 -> 14 -> 19 -> 2 -> 11 -> 13 -> 0
Distance: 222km

Longest Route: 236km
